# Setup

## Importando Bibliotecas

In [8]:
import pandas as pd
import duckdb
import numpy as np
import pickle
import os

from duckdb.duckdb import DuckDBPyConnection
from pandas import DataFrame

import gc
import psutil

final_conn = duckdb.connect('final_db')
big_conn = duckdb.connect('big_db')

## Funções Auxiliares

In [2]:
def sql(
    query:str,
    conn:DuckDBPyConnection=final_conn
) -> DataFrame:
    """
    Executa uma consulta SQL em uma conexão de banco duckDb e retorna o resultado em um dataframe Pandas.

    Parâmetros
    ----------

    query (str)
        Query SQL a ser realizada no banco
    
    conn (DuckDBPyConnection)
        Conexão com banco DB em que a consulta será realizada
    
    Returns
    ----------

    DataFrame: Dataframe do Pandas com o resultado da consulta SQL realizada

    """
    
    df = conn.execute(query).fetch_df()
    return df

def check_memory():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    return mem_info.rss / (1024 ** 2)  # Converte bytes para MB

## Importando bases

In [3]:
df_abt_base = sql("SELECT * FROM abt_base")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Escoragem

## Modelos 1

In [4]:
# Carregando Modelos
list_versions_1 = ['1_1', '1_2', '1_3', '1_4']
modelos_1 = {}

print("Importando modelos ...")
for version in list_versions_1:
    caminho = f'modelos_finais/modelo_{version}.pkl'
    try:
        with open(caminho, 'rb') as f:
            modelos_1[version] = pickle.load(f)
        
        print(f'✅ Versão {version} carregada com sucesso!')

    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho} não encontrado.')
    except Exception as e:
        print(f'⚠️ Erro inesperado na versão {version}: {e}')

# Escorando
print("\nRealizando a escoragem ...")

df_big_model_1 = df_abt_base.copy()

for version, model in modelos_1.items():
    try:
        if hasattr(model, 'feature_names_in_'):
            features = model.feature_names_in_
            df_big_model_1[f'vl_output_model_{version}'] = model.predict(df_big_model_1[features])
        else:
            df_big_model_1[f'vl_output_model_{version}'] = model.predict(df_big_model_1)
            
        print(f'✅ Coluna "vl_output_model_{version}" adicionada.')
        
    except Exception as e:
        print(f'❌ Erro ao processar versão {version}: {e}')

# Salvando a tabela big
print("\nEscrevendo a tabela big ...")
_ = sql("CREATE OR REPLACE TABLE big_model_1 AS (SELECT * FROM df_big_model_1)")

print("✅ Tabela big registrada com sucesso!\n")

# Liberando Memória
print("Liberando memória do dataframe big. . . ")
mem_antes = check_memory()
print(f"📊 Memória antes da limpeza: {mem_antes:.2f} MB")

del df_big_model_1
gc.collect()

mem_depois = check_memory()
print(f"🧹 Memória do DataFrame liberada com sucesso.")
print(f"📊 Memória após a limpeza: {mem_depois:.2f} MB")
print(f"📉 Economia real: {mem_antes - mem_depois:.2f} MB")

Importando modelos ...
✅ Versão 1_1 carregada com sucesso!
✅ Versão 1_2 carregada com sucesso!
✅ Versão 1_3 carregada com sucesso!
✅ Versão 1_4 carregada com sucesso!

Realizando a escoragem ...
✅ Coluna "vl_output_model_1_1" adicionada.
✅ Coluna "vl_output_model_1_2" adicionada.
✅ Coluna "vl_output_model_1_3" adicionada.
✅ Coluna "vl_output_model_1_4" adicionada.

Escrevendo a tabela big ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Tabela big registrada com sucesso!

Liberando memória do dataframe big. . . 
📊 Memória antes da limpeza: 14612.34 MB
🧹 Memória do DataFrame liberada com sucesso.
📊 Memória após a limpeza: 2828.68 MB
📉 Economia real: 11783.66 MB


## Modelos 2

In [6]:
# Carregando Modelos
list_versions_2 = ['2_1', '2_2', '2_3', '2_4']
modelos_2 = {}

print("Importando modelos ...")
for version in list_versions_2:
    caminho = f'modelos_finais/modelo_{version}.pkl'
    try:
        with open(caminho, 'rb') as f:
            modelos_2[version] = pickle.load(f)
        
        print(f'✅ Versão {version} carregada com sucesso!')

    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho} não encontrado.')
    except Exception as e:
        print(f'⚠️ Erro inesperado na versão {version}: {e}')

# Escorando
print("\nRealizando a escoragem ...")

df_big_model_2 = df_abt_base.copy()

for version, model in modelos_2.items():
    try:
        if hasattr(model, 'feature_names_in_'):
            features = model.feature_names_in_
            df_big_model_2[f'vl_output_model_{version}'] = model.predict(df_big_model_2[features])
        else:
            df_big_model_2[f'vl_output_model_{version}'] = model.predict(df_big_model_2)
            
        print(f'✅ Coluna "vl_output_model_{version}" adicionada.')
        
    except Exception as e:
        print(f'❌ Erro ao processar versão {version}: {e}')

# Salvando a tabela big
print("\nEscrevendo a tabela big ...")
_ = sql("CREATE OR REPLACE TABLE big_model_2 AS (SELECT * FROM df_big_model_2)")

print("✅ Tabela big registrada com sucesso!\n")

# Liberando Memória
print("Liberando memória do dataframe big. . . ")
mem_antes = check_memory()
print(f"📊 Memória antes da limpeza: {mem_antes:.2f} MB")

del df_big_model_2
gc.collect()

mem_depois = check_memory()
print(f"🧹 Memória do DataFrame liberada com sucesso.")
print(f"📊 Memória após a limpeza: {mem_depois:.2f} MB")
print(f"📉 Economia real: {mem_antes - mem_depois:.2f} MB")

Importando modelos ...
✅ Versão 2_1 carregada com sucesso!
✅ Versão 2_2 carregada com sucesso!
✅ Versão 2_3 carregada com sucesso!
✅ Versão 2_4 carregada com sucesso!

Realizando a escoragem ...
✅ Coluna "vl_output_model_2_1" adicionada.
✅ Coluna "vl_output_model_2_2" adicionada.
✅ Coluna "vl_output_model_2_3" adicionada.
✅ Coluna "vl_output_model_2_4" adicionada.

Escrevendo a tabela big ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Tabela big registrada com sucesso!

Liberando memória do dataframe big. . . 
📊 Memória antes da limpeza: 9424.73 MB
🧹 Memória do DataFrame liberada com sucesso.
📊 Memória após a limpeza: 2794.79 MB
📉 Economia real: 6629.95 MB


In [ ]:
df_abt_base['vl_prioridade_vizinha_1']

0           0.5627
1           0.5627
2           0.5627
3           0.5627
4           0.5627
             ...  
18144378    0.5322
18144379    0.5322
18144380    0.5322
18144381    0.5322
18144382    0.5322
Name: vl_prioridade_vizinha_1, Length: 18144383, dtype: float64

## Modelos 3

In [10]:
# Carregando Modelos
list_versions_3 = ['3_1', '3_2', '3_3', '3_4', '3_5']
modelos_3_especializado,modelos_3_geral = {},{}
dict_threshold_especializado = {
    '3_1':0.5,
    '3_2':0.75,
    '3_3':0.9,
    '3_4':0.4,
    '3_5':0.3
}

print("Importando modelos ...")
for version in list_versions_3:
    threshold = dict_threshold_especializado[version]
    caminho_especializado = f'modelos_finais/modelo_{version}_especializado.pkl'
    caminho_geral = f'modelos_finais/modelo_{version}_geral.pkl'
    try:
        with open(caminho, 'rb') as f:
            modelos_3_especializado[version] = pickle.load(f)
        
        print(f'✅ Versão {version} especializada carregada com sucesso!')

    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho_especializado} não encontrado.')
    try:
        with open(caminho_geral, 'rb') as f:
            modelos_3_geral[version] = pickle.load(f)
        
        print(f'✅ Versão {version} geral carregada com sucesso!')

    except FileNotFoundError:
        print(f'❌ Erro: Arquivo {caminho_geral} não encontrado.')

    except Exception as e:
        print(f'⚠️ Erro inesperado na versão {version}: {e}')

# Escorando
print("\nRealizando a escoragem ...")

df_big_model_3 = df_abt_base.copy()

for version in list_versions_3:
    modelo_especializado = modelos_3_especializado[version]
    modelo_geral = modelos_3_geral[version]

    try:
        features = model.feature_names_in_
        df_big_model_3[f'vl_output_model_especializado_{version}'] = modelo_especializado.predict(df_big_model_3[features])
        df_big_model_3[f'vl_output_model_geral_{version}'] = modelo_geral.predict(df_big_model_3[features])
        df_big_model_3[f'vl_output_model_{version}'] = np.where(
            df_big_model_3['vl_prioridade_vizinha_1']>=threshold,
            df_big_model_3[f'vl_output_model_especializado_{version}'],
            df_big_model_3[f'vl_output_model_geral_{version}']
            )
        df_big_model_3[f'vl_threshold_especializado_{version}'] = threshold

        print(f'✅ Colunas "vl_output_model_especializado_{version}", "vl_output_model_geral_{version}", "vl_threshold_especializado_{version}" e "vl_output_model_{version}" adicionadas.')
        
    except Exception as e:
        print(f'❌ Erro ao processar versão {version}: {e}')

# Salvando a tabela big
print("\nEscrevendo a tabela big ...")
_ = sql("CREATE OR REPLACE TABLE big_model_3 AS (SELECT * FROM df_big_model_3)")

print("✅ Tabela big registrada com sucesso!\n")

# Liberando Memória
print("Liberando memória do dataframe big. . . ")
mem_antes = check_memory()
print(f"📊 Memória antes da limpeza: {mem_antes:.2f} MB")

del df_big_model_3
gc.collect()

mem_depois = check_memory()
print(f"🧹 Memória do DataFrame liberada com sucesso.")
print(f"📊 Memória após a limpeza: {mem_depois:.2f} MB")
print(f"📉 Economia real: {mem_antes - mem_depois:.2f} MB")

Importando modelos ...
✅ Versão 3_1 especializada carregada com sucesso!
✅ Versão 3_1 geral carregada com sucesso!
✅ Versão 3_2 especializada carregada com sucesso!
✅ Versão 3_2 geral carregada com sucesso!
✅ Versão 3_3 especializada carregada com sucesso!
✅ Versão 3_3 geral carregada com sucesso!
✅ Versão 3_4 especializada carregada com sucesso!
✅ Versão 3_4 geral carregada com sucesso!
✅ Versão 3_5 especializada carregada com sucesso!
✅ Versão 3_5 geral carregada com sucesso!

Realizando a escoragem ...
✅ Colunas "vl_output_model_especializado_3_1", "vl_output_model_geral_3_1", "vl_threshold_especializado_3_1" e "vl_output_model_3_1" adicionadas.
✅ Colunas "vl_output_model_especializado_3_2", "vl_output_model_geral_3_2", "vl_threshold_especializado_3_2" e "vl_output_model_3_2" adicionadas.
✅ Colunas "vl_output_model_especializado_3_3", "vl_output_model_geral_3_3", "vl_threshold_especializado_3_3" e "vl_output_model_3_3" adicionadas.
✅ Colunas "vl_output_model_especializado_3_4", "vl_

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Tabela big registrada com sucesso!

Liberando memória do dataframe big. . . 
📊 Memória antes da limpeza: 2494.33 MB
🧹 Memória do DataFrame liberada com sucesso.
📊 Memória após a limpeza: 1939.52 MB
📉 Economia real: 554.82 MB
